## UPLOADING IN DRIVE

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving OnlineRetail.csv to OnlineRetail.csv


# LOADING THE DATASET

In [ ]:
import pandas as pd
df = pd.read_csv('OnlineRetail.csv', encoding='latin-1')



DATA EXPLORATION AND UNDERSTANDING

1.DATASET OVERVIEW

In [ ]:
# Shape of dataset
print("Dataset Shape:", df.shape)
print("\n")

# Column names and data types
print(df.info())
print("\n")

# First few rows
print(df.head(10))


Dataset Shape: (541909, 8)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
None


  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4

# 2.Check for Missing Values

In [ ]:
# Missing values count
print("Missing Values:\n")
print(df.isnull().sum())
print("\n")

# Missing values percentage
print("Missing Values Percentage:\n")
print(round(df.isnull().sum() / len(df) * 100, 2))


Missing Values:

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


Missing Values Percentage:

InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64


# 3: Statistical Summary

In [ ]:
# Statistical summary of numerical columns
print(df.describe())


            Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.690570
std       218.081158      96.759853    1713.600303
min    -80995.000000  -11062.060000   12346.000000
25%         1.000000       1.250000   13953.000000
50%         3.000000       2.080000   15152.000000
75%        10.000000       4.130000   16791.000000
max     80995.000000   38970.000000   18287.000000


# 4: Check for Duplicates

In [ ]:
# Check duplicates
print(f"Total Duplicate Rows: {df.duplicated().sum()}")


Total Duplicate Rows: 5268


# 5: Unique Values Check

In [ ]:
# Unique counts for key columns
print(f"Unique Invoices: {df['InvoiceNo'].nunique()}")
print(f"Unique Customers: {df['CustomerID'].nunique()}")
print(f"Unique Products: {df['StockCode'].nunique()}")
print(f"Unique Countries: {df['Country'].nunique()}")


Unique Invoices: 25900
Unique Customers: 4372
Unique Products: 4070
Unique Countries: 38


# 6:Data inspection

In [ ]:
print("="*50)
print("SAMPLE DATA FOR INSPECTION")
print("="*50)
print(df.sample(10))


SAMPLE DATA FOR INSPECTION
       InvoiceNo StockCode                          Description  Quantity  \
136744    547971     20975  12 PENCILS SMALL TUBE RED RETROSPOT         7   
8664      537141     82482    WOODEN PICTURE FRAME WHITE FINISH         6   
82340     543200     22139     RETROSPOT TEA SET CERAMIC 11 PC          2   
270304    560557     35653                VINTAGE BEAD NOTEBOOK         1   
125027    547005     22303             COFFEE MUG APPLES DESIGN         6   
262156    559902     84987      SET OF 36 TEATIME PAPER DOILIES         1   
132156    547656    47593A             CAROUSEL PONIES BABY BIB         2   
498206    578513     22423             REGENCY CAKESTAND 3 TIER         3   
223955    556497     22557     PLASTERS IN TIN VINTAGE PAISLEY         12   
94889     544434     21784                      SHOE SHINE BOX          1   

             InvoiceDate  UnitPrice  CustomerID         Country  
136744   3/28/2011 16:37       0.65     17917.0  United Kin

# DATA CLEANING

# Step 1: Convert StockCode to String

In [ ]:
# Convert StockCode to string to handle mixed types
df['StockCode'] = df['StockCode'].astype(str)

# Verify it's now all strings
print("StockCode dtype:", df['StockCode'].dtype)
print("\nStockCode sample values:")
print(df['StockCode'].head(20))


StockCode dtype: object

StockCode sample values:
0     85123A
1      71053
2     84406B
3     84029G
4     84029E
5      22752
6      21730
7      22633
8      22632
9      84879
10     22745
11     22748
12     22749
13     22310
14     84969
15     22623
16     22622
17     21754
18     21755
19     21777
Name: StockCode, dtype: object


# Step 2: Handle Missing Values

In [ ]:
# Check missing values again
print("Missing Values Before Cleaning:")
print(df.isnull().sum())
print("\n")

# Remove rows where CustomerID is missing (needed for segmentation)
df = df.dropna(subset=['CustomerID'])

# Remove rows where Description is missing
df = df.dropna(subset=['Description'])

print(f"Shape after removing NaN: {df.shape}")
print("\nMissing Values After Cleaning:")
print(df.isnull().sum())


Missing Values Before Cleaning:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


Shape after removing NaN: (406829, 8)

Missing Values After Cleaning:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


# Step 3: Remove Invalid Quantities

In [ ]:
# Check for negative or zero quantities
print(f"Rows with Quantity <= 0: {(df['Quantity'] <= 0).sum()}")
print(f"Rows with Quantity > 0: {(df['Quantity'] > 0).sum()}")

# Keep both positive and negative (negatives are returns - important for business)
# But remove zero quantity rows if any
df = df[df['Quantity'] != 0]

print(f"\nShape after removing zero quantities: {df.shape}")


Rows with Quantity <= 0: 8905
Rows with Quantity > 0: 397924

Shape after removing zero quantities: (406829, 8)


# Step 4: Remove Invalid Prices

In [ ]:
# Check for negative or zero prices
print(f"Rows with UnitPrice <= 0: {(df['UnitPrice'] <= 0).sum()}")

# Remove rows with zero or negative price (these are data errors)
df = df[df['UnitPrice'] > 0]

print(f"Shape after removing invalid prices: {df.shape}")


Rows with UnitPrice <= 0: 40
Shape after removing invalid prices: (406789, 8)


# Step 5: Remove Duplicates

In [ ]:
# Check duplicates
print(f"Duplicate rows before: {df.duplicated().sum()}")

# Remove duplicates
df = df.drop_duplicates()

print(f"Duplicate rows after: {df.duplicated().sum()}")
print(f"Final shape: {df.shape}")


Duplicate rows before: 5225
Duplicate rows after: 0
Final shape: (401564, 8)


# Step 6: Convert InvoiceDate to DateTime

In [ ]:
# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("InvoiceDate dtype:", df['InvoiceDate'].dtype)
print("\nInvoiceDate range:")
print(f"Start: {df['InvoiceDate'].min()}")
print(f"End: {df['InvoiceDate'].max()}")


InvoiceDate dtype: datetime64[ns]

InvoiceDate range:
Start: 2010-12-01 08:26:00
End: 2011-12-09 12:50:00


# FEATURE ENGINEERING

# Step 1: Create Revenue Column

In [ ]:
# Calculate Revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

print("Revenue column created")
print(df[['Quantity', 'UnitPrice', 'Revenue']].head(10))
print(f"\nRevenue statistics:")
print(df['Revenue'].describe())


Revenue column created
   Quantity  UnitPrice  Revenue
0         6       2.55    15.30
1         6       3.39    20.34
2         8       2.75    22.00
3         6       3.39    20.34
4         6       3.39    20.34
5         2       7.65    15.30
6         6       4.25    25.50
7         6       1.85    11.10
8         6       1.85    11.10
9        32       1.69    54.08

Revenue statistics:
count    401564.000000
mean         20.615691
std         430.373603
min     -168469.600000
25%           4.250000
50%          11.700000
75%          19.800000
max      168469.600000
Name: Revenue, dtype: float64


# Step 2: Extract Date Features

In [ ]:
# Extract date components
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Week'] = df['InvoiceDate'].dt.isocalendar().week
df['Quarter'] = df['InvoiceDate'].dt.quarter

print("Date features created")
print(df[['InvoiceDate', 'Year', 'Month', 'Day', 'DayOfWeek', 'Week', 'Quarter']].head(10))


Date features created
          InvoiceDate  Year  Month  Day  DayOfWeek  Week  Quarter
0 2010-12-01 08:26:00  2010     12    1  Wednesday    48        4
1 2010-12-01 08:26:00  2010     12    1  Wednesday    48        4
2 2010-12-01 08:26:00  2010     12    1  Wednesday    48        4
3 2010-12-01 08:26:00  2010     12    1  Wednesday    48        4
4 2010-12-01 08:26:00  2010     12    1  Wednesday    48        4
5 2010-12-01 08:26:00  2010     12    1  Wednesday    48        4
6 2010-12-01 08:26:00  2010     12    1  Wednesday    48        4
7 2010-12-01 08:28:00  2010     12    1  Wednesday    48        4
8 2010-12-01 08:28:00  2010     12    1  Wednesday    48        4
9 2010-12-01 08:34:00  2010     12    1  Wednesday    48        4


# Step 3: Create Transaction Type (Sale vs Return)

In [ ]:
# Identify returns (negative quantity = returns)
df['TransactionType'] = df['Quantity'].apply(lambda x: 'Return' if x < 0 else 'Sale')

print("Transaction Type created")
print(df['TransactionType'].value_counts())


Transaction Type created
TransactionType
Sale      392692
Return      8872
Name: count, dtype: int64


# Step 4: Create Customer Lifetime Metrics (for RFM)

In [ ]:
# Calculate for each customer:
# - Total spending (Monetary)
# - Number of transactions (Frequency)
# - Last purchase date (for Recency calculation)

customer_metrics = df.groupby('CustomerID').agg({
    'Revenue': 'sum',  # Total spending
    'InvoiceNo': 'count',  # Number of transactions
    'InvoiceDate': 'max'  # Last purchase date
}).reset_index()

customer_metrics.columns = ['CustomerID', 'Total_Spending', 'Transaction_Count', 'Last_Purchase_Date']

print("Customer metrics created")
print(customer_metrics.head(10))
print(f"\nShape: {customer_metrics.shape}")


Customer metrics created
   CustomerID  Total_Spending  Transaction_Count  Last_Purchase_Date
0     12346.0            0.00                  2 2011-01-18 10:17:00
1     12347.0         4310.00                182 2011-12-07 15:52:00
2     12348.0         1797.24                 31 2011-09-25 13:13:00
3     12349.0         1757.55                 73 2011-11-21 09:51:00
4     12350.0          334.40                 17 2011-02-02 16:01:00
5     12352.0         1545.41                 95 2011-11-03 14:37:00
6     12353.0           89.00                  4 2011-05-19 17:47:00
7     12354.0         1079.40                 58 2011-04-21 13:11:00
8     12355.0          459.40                 13 2011-05-09 13:49:00
9     12356.0         2811.43                 59 2011-11-17 08:40:00

Shape: (4371, 4)


# Step 5: Calculate Recency

In [ ]:
# Calculate recency (days since last purchase from max date in dataset)
reference_date = df['InvoiceDate'].max()

customer_metrics['Recency_Days'] = (reference_date - customer_metrics['Last_Purchase_Date']).dt.days

print("Recency calculated")
print(customer_metrics[['CustomerID', 'Last_Purchase_Date', 'Recency_Days']].head(10))


Recency calculated
   CustomerID  Last_Purchase_Date  Recency_Days
0     12346.0 2011-01-18 10:17:00           325
1     12347.0 2011-12-07 15:52:00             1
2     12348.0 2011-09-25 13:13:00            74
3     12349.0 2011-11-21 09:51:00            18
4     12350.0 2011-02-02 16:01:00           309
5     12352.0 2011-11-03 14:37:00            35
6     12353.0 2011-05-19 17:47:00           203
7     12354.0 2011-04-21 13:11:00           231
8     12355.0 2011-05-09 13:49:00           213
9     12356.0 2011-11-17 08:40:00            22


# Step 6: Rename Columns for RFM

In [ ]:
# Rename for clarity in RFM analysis
customer_metrics.columns = ['CustomerID', 'Monetary', 'Frequency', 'Last_Purchase_Date', 'Recency']

print("RFM columns ready:")
print(customer_metrics.head(10))


RFM columns ready:
   CustomerID  Monetary  Frequency  Last_Purchase_Date  Recency
0     12346.0      0.00          2 2011-01-18 10:17:00      325
1     12347.0   4310.00        182 2011-12-07 15:52:00        1
2     12348.0   1797.24         31 2011-09-25 13:13:00       74
3     12349.0   1757.55         73 2011-11-21 09:51:00       18
4     12350.0    334.40         17 2011-02-02 16:01:00      309
5     12352.0   1545.41         95 2011-11-03 14:37:00       35
6     12353.0     89.00          4 2011-05-19 17:47:00      203
7     12354.0   1079.40         58 2011-04-21 13:11:00      231
8     12355.0    459.40         13 2011-05-09 13:49:00      213
9     12356.0   2811.43         59 2011-11-17 08:40:00       22


# Step 7: Create Product Analysis Features

In [ ]:
# Category from Description (extract first word or category)
df['Category'] = df['Description'].str.split().str[0]

# Product quantity (total units sold per product)
product_metrics = df.groupby('StockCode').agg({
    'Description': 'first',
    'Revenue': 'sum',
    'Quantity': 'sum',
    'InvoiceNo': 'count'
}).reset_index()

product_metrics.columns = ['StockCode', 'Product_Description', 'Total_Revenue', 'Total_Quantity', 'Transaction_Count']

print("Product metrics created")
print(product_metrics.head(10))


Product metrics created
  StockCode           Product_Description  Total_Revenue  Total_Quantity  \
0     10002   INFLATABLE POLITICAL GLOBE          699.55             823   
1     10080      GROOVY CACTUS INFLATABLE         114.41             291   
2     10120                  DOGGY RUBBER          40.32             192   
3    10123C         HEARTS WRAPPING TAPE            3.25               5   
4    10124A   SPOTS ON RED BOOKCOVER TAPE           6.72              16   
5    10124G      ARMY CAMO BOOKCOVER TAPE           7.14              17   
6     10125       MINI FUNKY DESIGN TAPES         929.45            1225   
7     10133  COLOURING PENCILS BROWN TUBE        1134.79            2363   
8     10135  COLOURING PENCILS BROWN TUBE        1782.94            1935   
9     11001   ASSTD DESIGN RACING CAR PEN        1715.95            1067   

   Transaction_Count  
0                 49  
1                 21  
2                 29  
3                  3  
4                  5  
5

# Step 8: Verify Cleaned & Featured Data

In [ ]:
print("="*50)
print("FINAL CLEANED DATASET SUMMARY")
print("="*50)
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())


FINAL CLEANED DATASET SUMMARY
Shape: (401564, 17)

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Year', 'Month', 'Day', 'DayOfWeek', 'Week', 'Quarter', 'TransactionType', 'Category']

First 5 rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  Revenue  Year  \
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom    15.30  2010   
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom    20.34  2010   
2 2010-12-01 08:26:00       2.75     17850.0  Unit

DOWNLOAD THE CLEANED DATASET

In [ ]:
df.to_csv('cleaned.csv', index=False)
from google.colab import files
files.download('cleaned.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>